In [ ]:
from fastf1.ergast import Ergast
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
import json

YEAR = 2026

ergast = Ergast()
DRIVERS = ergast.get_driver_info(YEAR)["driverId"].to_list()

schedule = fastf1.get_event_schedule(YEAR)
races = schedule["RoundNumber"][schedule["RoundNumber"] > 0].tolist()

# already_processed = {i: 0 for i in races}
# race_replays = [f for f in os.listdir(f'data/{YEAR}/replays') if f.startswith('telemetry_')]
# for file in race_replays:
#     race_id_str = file.split(f'_{YEAR}_')[1].split('.')[0]
#     race_id = int(race_id_str)
#     already_processed[race_id] += 1

# last_race = 0
# for k, v in already_processed.items():
#     if v == len(DRIVERS):
#         last_race = k
# next_race = last_race + 1 if last_race + 1 < len(races) else None

# if not next_race:
#     raise Exception("All races have been processed.")

# unprocessed_races = [race for race in races if race >= next_race]

unprocessed_races = [races[0]] # FIXME: temporary

for race in unprocessed_races:
    
    print(f"Processing race {race}...")
    
    session = fastf1.get_session(YEAR, race, "R")
    session.load(laps=True, telemetry=True, weather=False)
    lap_replays = [[] for _ in range(session.total_laps)]
    
    for idx, driver in enumerate(DRIVERS):
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        
        print(f"\tProcessing driver [{idx+1}/{len(DRIVERS)}] {driver_slug}...........", end="")
        
        best_sectors = [float('inf'), float('inf'), float('inf')]
        best_lap_time = float('inf')
        
        for driver_lap in driver_laps.iterlaps():            
            lap_telemetry = driver_lap[1].get_telemetry()
            lap_number = int(driver_lap[1]['LapNumber'])
            
            best_sectors = [
                min(best_sectors[0], driver_lap[1]['Sector1Time'].total_seconds()),
                min(best_sectors[1], driver_lap[1]['Sector2Time'].total_seconds()),
                min(best_sectors[2], driver_lap[1]['Sector3Time'].total_seconds()),
            ]
            
            best_sectors = [0.0 if bs == float('inf') else bs for bs in best_sectors]
            
            best_lap_time = min(best_lap_time, driver_lap[1]['LapTime'].total_seconds())
            
            for _, row in lap_telemetry.iterrows():   
                lap_replays[lap_number - 1].append({
                    "driver": driver_slug,
                    "lap_number": lap_number,
                    "x": round(row['X'], 2),
                    "y": round(row['Y'], 2),
                    "z": round(row['Z'], 2),
                    "time": row['Time'].total_seconds(),
                    "position": driver_lap[1]['Position'], 
                    "compound": driver_lap[1]['Compound'],
                    "stint": driver_lap[1]['TyreLife'],
                    "gap_to_leader": 0, # TODO:
                    "gap_to_front": 0, # TODO:
                    "current_best_lap_time": best_lap_time,
                    "last_lap_time": driver_lap[1]['LapTime'].total_seconds(),
                    "current_sector_times": [
                        driver_lap[1]['Sector1Time'].total_seconds() if driver_lap[1]['Sector1Time'].total_seconds() > 0 else 0.0,
                        driver_lap[1]['Sector2Time'].total_seconds() if driver_lap[1]['Sector2Time'].total_seconds() > 0 else 0.0,
                        driver_lap[1]['Sector3Time'].total_seconds() if driver_lap[1]['Sector3Time'].total_seconds() > 0 else 0.0,
                    ],
                    "best_sector_time": best_sectors,
                    "is_in_pit": type(driver_lap[1]["PitInTime"].total_seconds()) is float, # TODO:
                    "is_retired": False, # TODO:
                })
        
        print(" [done]")
        
    os.makedirs(f'data/{YEAR}/replays/race_{race}', exist_ok=True)
    for idx, lap_replay in enumerate(lap_replays):
        lap_replay.sort(key=lambda x: x['time'])
        filename = f'data/{YEAR}/replays/race_{race}/lap_{idx+1}.json'
        with open(filename, 'w') as f:
            json.dump(lap_replay, f)


Processing race 1...
	Processing driver [1/30] albon_2026........... [done]
	Processing driver [2/30] alonso_2026........... [done]
	Processing driver [3/30] antonelli_2026........... [done]
	Processing driver [4/30] paul_aron_2026........... [done]
	Processing driver [5/30] bearman_2026........... [done]
	Processing driver [6/30] dino_beganovic_2026........... [done]
	Processing driver [7/30] bortoleto_2026........... [done]
	Processing driver [8/30] bottas_2026........... [done]
	Processing driver [9/30] luke_browning_2026........... [done]
	Processing driver [10/30] colapinto_2026........... [done]
	Processing driver [11/30] jak_crawford_2026........... [done]
	Processing driver [12/30] leonardo_fornaroli_2026........... [done]
	Processing driver [13/30] gasly_2026........... [done]
	Processing driver [14/30] hadjar_2026........... [done]
	Processing driver [15/30] hamilton_2026........... [done]
	Processing driver [16/30] colton_herta_2026........... [done]
	Processing driver [17/3